> **SUPERSEDED — DO NOT CITE.**
>
> All outputs below predate the Phase 1.4 recalibration: demand base
> 23.08 → 37.09 TWh (the old figure was *collected* energy, a revenue
> quantity), solar CF 0.27 → 0.20, gas deliverability re-anchored to
> 89.27 TWh_th. Baseline NPV moved from $5.98bn to $16.14bn.
>
> `01_data_validation` additionally validates against the superseded 23.08
> base and the 0.27 CF, so its *structure* is reusable but none of its
> assertions are. `04` was never run to completion.
>
> Retained for provenance. Rebuild after Phase 2.6 (genset backstop), because
> any run combining a binding NDC cap with a binding capital envelope is
> currently VoLL-dominated and not economically interpretable.

# 04 — Reliability Feasibility and the Financing Frontier

**Purpose:** Answers the reliability research questions (REL-1, REL-2, REL-3) and directly addresses the prime research question of the merged thesis: *how do capital structure and financing mechanism reshape the reliability feasibility frontier?*

**Prerequisite scripts:** `09_run_rel1_feasibility.py`, `10_run_rel2_marginal_cost.py`, `11_run_rel3_financing_frontier.py`  
**Data loaded from:** `results/rel1/`, `results/rel2/`, `results/rel3/`

---

**Research questions addressed:**

- **REL-1:** Does the feasibility of minimum reliability standards depend primarily on gas deliverability regime rather than solar build rate? *(Primary determinant hypothesis)*
- **REL-2:** What is the marginal cost of tightening the reliability standard, and how does this curve shift across gas regimes?
- **REL-3 (Merged thesis centrepiece):** Does EaaS financing improve the reliability feasibility frontier relative to public-capital-only financing — and does this improvement grow under NDC 3.0?

**Key concept — feasibility frontier:**  
For each financing arm and gas regime, there is a tightest reliability standard (smallest unserved fraction ε) that can still be achieved. Below this threshold, the system is infeasible — no amount of solar deployment can simultaneously meet the NDC cap, serve demand, and satisfy the reliability constraint. EaaS shifts this threshold by expanding the feasible space.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

ROOT = Path(".").resolve().parent
sys.path.append(str(ROOT))

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 200,
})

FIGURES_DIR = ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REL1_DIR = ROOT / "results" / "rel1"
REL2_DIR = ROOT / "results" / "rel2"
REL3_DIR = ROOT / "results" / "rel3"

GAS_COLORS = {
    "baseline":       "#4CAF50",
    "upside":         "#2196F3",
    "downside":       "#FF5722",
    "shock_recovery": "#9C27B0",
}

POLICY_ORDER = ["no_policy", "ndc2_unconditional", "ndc2_conditional",
                "ndc3_unconditional", "ndc3_conditional"]
POLICY_LABELS = {
    "no_policy":         "No Policy",
    "ndc2_unconditional": "NDC2 Uncond.",
    "ndc2_conditional":   "NDC2 Cond.",
    "ndc3_unconditional": "NDC3 Uncond.",
    "ndc3_conditional":   "NDC3 Cond.",
}

print(f"Repo root: {ROOT}")

## Section 1 — REL-1: Primary Determinant of Reliability Feasibility

In [ ]:
# ── Load REL-1 data ────────────────────────────────────────
feasibility_matrix = pd.read_csv(REL1_DIR / "feasibility_matrix.csv")
feasibility_threshold = pd.read_csv(REL1_DIR / "feasibility_threshold.csv")

with open(REL1_DIR / "rel1_summary.json") as f:
    rel1_summary = json.load(f)

print("Primary determinant:", rel1_summary["primary_determinant"])
print("Max gas range (eps):", rel1_summary["max_gas_range_eps"])
print("Max solar diff (eps):", rel1_summary["max_solar_diff_eps"])
print("\n", rel1_summary["interpretation"])

In [ ]:
# ── Table: Feasibility threshold by gas regime and solar build ─
thresh_display = feasibility_threshold[[
    "gas_case", "solar_build_case",
    "feasibility_threshold_eps", "fully_infeasible",
    "npv_cost_at_threshold"
]].copy()

thresh_display["feasibility_threshold_eps"] = thresh_display["feasibility_threshold_eps"].apply(
    lambda x: f"{x:.2f}" if pd.notna(x) else "ALL INFEASIBLE"
)
thresh_display["Reliability at threshold"] = feasibility_threshold["feasibility_threshold_eps"].apply(
    lambda x: f"{(1-x)*100:.0f}%" if pd.notna(x) else "—"
)
thresh_display["npv_cost_at_threshold"] = thresh_display["npv_cost_at_threshold"].apply(
    lambda x: f"${x/1e9:.1f}B" if pd.notna(x) else "—"
)
thresh_display = thresh_display.rename(columns={
    "gas_case": "Gas regime",
    "solar_build_case": "Solar build",
    "feasibility_threshold_eps": "Threshold ε",
    "fully_infeasible": "Fully infeasible",
    "npv_cost_at_threshold": "Cost at threshold",
})

display(thresh_display.style.hide(axis="index").set_caption(
    "Table 1: REL-1 Feasibility Threshold — Gas Regime vs Solar Build Rate"))

In [ ]:
# ── Figure: Feasibility threshold by gas regime ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

solar_build_cases = feasibility_threshold["solar_build_case"].unique()

for idx, solar_case in enumerate(solar_build_cases):
    ax = axes[idx]
    sub = feasibility_threshold[feasibility_threshold["solar_build_case"] == solar_case].copy()
    sub = sub.sort_values("feasibility_threshold_eps", ascending=False, na_position="last")

    gas_cases = sub["gas_case"].values
    thresholds = sub["feasibility_threshold_eps"].values
    reliabilities = [1 - t if pd.notna(t) else None for t in thresholds]

    colors = [GAS_COLORS.get(g, "gray") for g in gas_cases]

    bar_vals = [r * 100 if r is not None else 0 for r in reliabilities]
    infeas_mask = [t is None or pd.isna(t) for t in thresholds]

    bars = ax.barh(range(len(gas_cases)), bar_vals, color=colors, alpha=0.8)

    for i, (bar, infeas) in enumerate(zip(bars, infeas_mask)):
        if infeas:
            ax.text(5, i, "INFEASIBLE", va="center", fontsize=9,
                    color="red", fontweight="bold")
        else:
            ax.text(bar.get_width() + 0.3, i,
                    f"{bar.get_width():.0f}%",
                    va="center", fontsize=9)

    ax.set_yticks(range(len(gas_cases)))
    ax.set_yticklabels(gas_cases, fontsize=10)
    ax.set_xlabel("Tightest Achievable Reliability (%)")
    ax.set_title(f"Solar build: {solar_case}")
    ax.set_xlim(0, 110)
    ax.axvline(x=95, color="black", linestyle="--", alpha=0.4, label="95% benchmark")
    ax.grid(axis="x", alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle(
    "REL-1: Tightest Achievable Reliability by Gas Regime\n"
    "(Does gas regime determine feasibility more than solar build rate?)",
    fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rel1_feasibility_threshold.png", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "rel1_feasibility_threshold.pdf", bbox_inches="tight")
plt.show()
print("Saved: rel1_feasibility_threshold.png")

**REL-1 Interpretation:**

If the threshold varies substantially across gas regimes (left panel vs right panel shows small difference) but little across solar build rates, gas deliverability is the primary determinant. This validates the hypothesis that Nigeria's reliability problem is fundamentally a fuel supply problem, not a capital deployment problem. Increasing solar build rate does not rescue reliability when gas is scarce — you need either more gas or a structural alternative (EaaS + storage).

## Section 2 — REL-2: Marginal Cost of Reliability

In [ ]:
# ── Load REL-2 data ────────────────────────────────────────
cost_curve_all = pd.read_csv(REL2_DIR / "cost_curve_all.csv")
cost_curve_binding = pd.read_csv(REL2_DIR / "cost_curve_binding.csv")
curve_summary = pd.read_csv(REL2_DIR / "curve_summary.csv")

with open(REL2_DIR / "rel2_summary.json") as f:
    rel2_summary = json.load(f)

print("Cost curve shape:", cost_curve_all.shape)
print("Binding region rows:", len(cost_curve_binding))
print("\nCurve summary:")
print(curve_summary[["gas_case", "onset_eps", "onset_reliability",
                       "n_binding_eps"]].to_string(index=False))

In [ ]:
# ── Figure: Reliability cost curves ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Cost vs reliability (cost curve)
ax = axes[0]
feasible = cost_curve_all[
    (cost_curve_all["status"] == "feasible") &
    (cost_curve_all["eps"].notna())
].copy()

for gas_case, color in GAS_COLORS.items():
    sub = feasible[feasible["gas_case"] == gas_case].sort_values("eps", ascending=False)
    if len(sub):
        reliability_vals = (1 - sub["eps"]) * 100
        ax.plot(reliability_vals, sub["npv_cost"] / 1e9,
                "o-", color=color, label=gas_case, linewidth=2, markersize=6)

ax.set_xlabel("Reliability Standard (% served)")
ax.set_ylabel("NPV System Cost ($B)")
ax.set_title("Reliability-Cost Curve by Gas Regime")
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Marginal cost (dual) vs reliability in binding region
ax = axes[1]
for gas_case, color in GAS_COLORS.items():
    sub = cost_curve_binding[
        cost_curve_binding["gas_case"] == gas_case
    ].sort_values("eps", ascending=False)
    if len(sub) and sub["dual_usd_per_pct_pt"].notna().any():
        reliability_vals = (1 - sub["eps"]) * 100
        dual_vals = sub["dual_usd_per_pct_pt"].values / 1e6
        ax.plot(reliability_vals, dual_vals,
                "s-", color=color, label=gas_case, linewidth=2, markersize=7)

ax.set_xlabel("Reliability Standard (% served)")
ax.set_ylabel("Marginal Cost (M USD per percentage point)")
ax.set_title("Marginal Cost of Reliability (Binding Region Only)")
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle(
    "REL-2: Reliability Cost Curves — How Steeply Does Reliability Cost Rise?",
    fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rel2_reliability_cost_curves.png", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "rel2_reliability_cost_curves.pdf", bbox_inches="tight")
plt.show()
print("Saved: rel2_reliability_cost_curves.png")

In [ ]:
# ── Table: Curve onset and left-shift ─────────────────────
shift_info = rel2_summary.get("left_shift_analysis", {})
shift_rows = []
for gas_case, info in shift_info.items():
    onset = info.get("onset_eps")
    onset_rel = 1 - onset if onset is not None else None
    left_shift = info.get("left_shift_eps")
    shifted = info.get("curve_left_shifted")
    shift_rows.append({
        "Gas regime": gas_case,
        "Onset ε (binding starts)": f"{onset:.2f}" if onset else "never binds",
        "Onset reliability": f"{onset_rel*100:.0f}%" if onset_rel else "—",
        "Left shift vs upside": f"{left_shift:.2f}" if left_shift is not None else "0.00",
        "Curve left-shifted": "Yes" if shifted else "No",
    })

display(pd.DataFrame(shift_rows).style.hide(axis="index").set_caption(
    "Table 2: REL-2 Curve Onset and Left-Shift Analysis"))

if rel2_summary.get("interpretation"):
    print("\nSummary interpretation:")
    print(rel2_summary["interpretation"])

**REL-2 Interpretation:**

A left-shifted curve means the reliability constraint becomes binding at a looser standard (higher unserved fraction) — reliability becomes costly earlier. Under the downside gas regime, the curve should be most left-shifted: even a 80% reliability standard is expensive because gas scarcity forces costly solar deployment to compensate. Under upside gas, the constraint stays slack until much tighter standards are imposed.

The steepness comparison tells you how much each percentage point of reliability improvement costs. Under adverse gas conditions, steep curves mean reliability is expensive at the margin — each additional point of reliability requires disproportionately more solar investment.

## Section 3 — REL-3: EaaS and the Financing Frontier

This is the centrepiece of the merged thesis. REL-3 directly answers the prime research question:  
*how does financing structure (public-only vs EaaS) reshape the reliability feasibility frontier?*

In [ ]:
# ── Load REL-3 data ────────────────────────────────────────
frontier_all = pd.read_csv(REL3_DIR / "frontier_all.csv")
frontier_threshold = pd.read_csv(REL3_DIR / "frontier_threshold.csv")
frontier_shift = pd.read_csv(REL3_DIR / "frontier_shift.csv")

with open(REL3_DIR / "rel3_summary.json") as f:
    rel3_summary = json.load(f)

print("Frontier shift shape:", frontier_shift.shape)
print("Policy labels:", frontier_shift["policy_label"].unique())
print("Gas cases:", frontier_shift["gas_case"].unique())

In [ ]:
# ── Table: Frontier shift by gas regime and policy ─────────
for policy in POLICY_ORDER:
    pol_shift = frontier_shift[
        frontier_shift["policy_label"] == policy
    ].copy()
    if len(pol_shift) == 0:
        continue

    shift_rows = []
    for _, row in pol_shift.iterrows():
        pub_thresh = row["pub_threshold_eps"]
        eaas_thresh = row["eaas_threshold_eps"]
        shift = row["frontier_shift_eps"]
        shift_rows.append({
            "Gas regime": row["gas_case"],
            "Public threshold ε": f"{pub_thresh:.2f}" if pd.notna(pub_thresh) else "INFEASIBLE",
            "EaaS threshold ε": f"{eaas_thresh:.2f}" if pd.notna(eaas_thresh) else "INFEASIBLE",
            "Frontier shift": f"+{shift:.2f}" if (pd.notna(shift) and shift > 0) else
                              (f"{shift:.2f}" if pd.notna(shift) else "n/a"),
            "EaaS better": "YES" if row["eaas_strictly_better"] else "no",
            "EaaS necessary": "YES" if row["eaas_necessary"] else "no",
        })

    pol_label = POLICY_LABELS.get(policy, policy)
    display(pd.DataFrame(shift_rows).style.hide(axis="index").set_caption(
        f"Table 3: REL-3 Frontier Shift — {pol_label}"))
    print()

In [ ]:
# ── Figure: Frontier shift heatmap ─────────────────────────
# Pivot: gas_case × policy_label, values = frontier_shift_eps
pivot_shift = frontier_shift.pivot_table(
    index="gas_case",
    columns="policy_label",
    values="frontier_shift_eps",
    aggfunc="first"
)

gas_order = ["downside", "shock_recovery", "baseline", "upside"]
pol_order_avail = [p for p in POLICY_ORDER if p in pivot_shift.columns]
gas_order_avail = [g for g in gas_order if g in pivot_shift.index]

pivot_shift = pivot_shift.loc[gas_order_avail, pol_order_avail]
pivot_shift.columns = [POLICY_LABELS.get(c, c) for c in pivot_shift.columns]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(
    pivot_shift.values.astype(float),
    cmap="RdYlGn", aspect="auto",
    vmin=-0.05, vmax=0.15
)

ax.set_xticks(range(len(pivot_shift.columns)))
ax.set_xticklabels(pivot_shift.columns, rotation=25, ha="right", fontsize=10)
ax.set_yticks(range(len(pivot_shift.index)))
ax.set_yticklabels(pivot_shift.index, fontsize=11)

for i in range(len(pivot_shift.index)):
    for j in range(len(pivot_shift.columns)):
        val = pivot_shift.values[i, j]
        if not np.isnan(val):
            label = f"+{val:.2f}" if val > 0 else f"{val:.2f}"
            ax.text(j, i, label,
                    ha="center", va="center",
                    fontsize=10, fontweight="bold",
                    color="white" if abs(val) > 0.08 else "black")
        else:
            ax.text(j, i, "n/a", ha="center", va="center",
                    fontsize=9, color="gray")

plt.colorbar(im, ax=ax, label="Frontier shift (ε units, positive = EaaS better)")
ax.set_title(
    "REL-3: EaaS Frontier Improvement — How Much Does Financing Structure Expand Feasible Space?",
    fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rel3_frontier_shift_heatmap.png", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "rel3_frontier_shift_heatmap.pdf", bbox_inches="tight")
plt.show()
print("Saved: rel3_frontier_shift_heatmap.png")

In [ ]:
# ── Figure: Frontier curves for key gas × policy combinations ─
# Shows the actual cost-reliability frontier for public vs EaaS
# Key combination: downside gas under ndc3_unconditional

key_combos = [
    ("downside", "ndc3_unconditional"),
    ("baseline", "ndc3_unconditional"),
    ("downside", "no_policy"),
]

n_combos = len(key_combos)
fig, axes = plt.subplots(1, n_combos, figsize=(6 * n_combos, 6), sharey=False)
if n_combos == 1:
    axes = [axes]

for idx, (gas_case, policy) in enumerate(key_combos):
    ax = axes[idx]

    sub = frontier_all[
        (frontier_all["gas_case"] == gas_case) &
        (frontier_all["policy_label"] == policy) &
        (frontier_all["status"] == "feasible") &
        (frontier_all["eps"].notna())
    ].copy()

    for fin_label, (color, ls) in [
        ("public_only", ("#FF5722", "--")),
        ("eaas", ("#2196F3", "-")),
    ]:
        # Handle different column names for financing label
        fin_col = "financing_label" if "financing_label" in sub.columns else "financing_arm"
        sub_fin = sub[sub[fin_col] == fin_label].sort_values("eps", ascending=False)
        if len(sub_fin):
            reliability_pct = (1 - sub_fin["eps"]) * 100
            ax.plot(reliability_pct, sub_fin["npv_cost"] / 1e9,
                    "o-", color=color, linestyle=ls,
                    label=fin_label.replace("_", "-"),
                    linewidth=2, markersize=7)

    ax.set_xlabel("Reliability Standard (% served)")
    ax.set_ylabel("NPV System Cost ($B)")
    ax.set_title(f"Gas: {gas_case}\nPolicy: {POLICY_LABELS.get(policy, policy)}",
                 fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle(
    "REL-3: Reliability Feasibility Frontier — Public vs EaaS Financing",
    fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rel3_frontier_curves.png", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "rel3_frontier_curves.pdf", bbox_inches="tight")
plt.show()
print("Saved: rel3_frontier_curves.png")

In [ ]:
# ── NDC relevance: Does EaaS improvement grow with ambition? ─
ndc_rel = rel3_summary.get("ndc_relevance", {})

ndc_rows = [
    {"Policy": "No policy",
     "Max frontier shift": ndc_rel.get("max_shift_no_policy")},
    {"Policy": "NDC3 unconditional",
     "Max frontier shift": ndc_rel.get("max_shift_ndc_unconditional")},
    {"Policy": "NDC3 conditional",
     "Max frontier shift": ndc_rel.get("max_shift_ndc_conditional")},
]
ndc_df = pd.DataFrame(ndc_rows)
ndc_df["Max frontier shift"] = ndc_df["Max frontier shift"].apply(
    lambda x: f"+{x:.3f}" if x is not None else "n/a"
)

display(ndc_df.style.hide(axis="index").set_caption(
    "Table 4: Does EaaS Frontier Improvement Grow with NDC Ambition?"))

grows = ndc_rel.get("eaas_improvement_grows_with_ndc_ambition")
print(f"\nEaaS improvement grows with NDC ambition: {grows}")
print(ndc_rel.get("interpretation", ""))

**REL-3 Interpretation — The Merged Thesis Centrepiece:**

The frontier shift heatmap directly answers the prime research question. A positive shift means EaaS achieves tighter reliability standards that public capital cannot — the feasible space expands when financing is restructured.

Three findings matter most for the merged thesis:

**Finding 1 — Gas regime dependence:** The shift should be largest under the downside gas regime, where public capital is most severely exhausted by the combined reliability and emissions compliance requirements. Under upside gas, public capital alone may be sufficient.

**Finding 2 — NDC amplification:** If the shift grows from no_policy → ndc3_unconditional → ndc3_conditional, EaaS becomes progressively more important as climate ambition increases. This directly supports the thesis claim that EaaS is a structural hedge against international finance non-arrival.

**Finding 3 — EaaS necessity:** Where `eaas_necessary = YES`, EaaS is not merely convenient — it is the only mechanism that makes a reliability standard achievable. These cells in the heatmap are the strongest evidence for the EaaS contribution.

---

## Summary

**REL-1:** The primary determinant of reliability feasibility is [gas regime / solar build rate — read from summary]. The gas regime shifts the threshold by [X] ε units; varying solar build rate shifts it by only [Y] ε units. This confirms Nigeria's reliability problem is fundamentally a fuel supply problem.

**REL-2:** The marginal cost curve is left-shifted under adverse gas regimes — reliability becomes costly at looser standards when gas is scarce. The downside regime shows the steepest curve, meaning each percentage point of reliability improvement is most expensive precisely when fuel supply is most constrained.

**REL-3:** EaaS shifts the reliability frontier by up to [X] ε units under [gas_case + policy]. The shift grows with NDC ambition: [confirm or deny from Table 4]. EaaS is necessary (not merely convenient) in [N] gas–policy combinations. This is the direct answer to the prime research question: financing structure materially reshapes the feasibility frontier under gas-constrained NDC compliance.